**1. Importación de librerías**

En esta celda se importan las librerías necesarias para ejecutar los distintos procesos del proyecto.

In [1]:
import cv2
import csv
import math 
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import json
import random
import shutil
from pathlib import Path
import torch
import easyocr

**2. Detección y análisis del sentido de circulación**

Se realiza la detección y el seguimiento de personas, bicicletas y coches mediante YOLO11n. A partir de sus movimientos respecto a unas líneas de referencia, se contabilizan los objetos que se acercan y se alejan.

In [2]:
#Se carga del modelo YOLO y se definen de las clases que se van a detectar
model = YOLO("yolo11n.pt")

classNames = ["person", "bicycle", "car"]

cap = cv2.VideoCapture("C0142.MP4")

#Se obtienen las características del vídeo original
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

#Se crea el vídeo de salida con las mismas dimensiones y FPS que el original
out = cv2.VideoWriter("C0142_direccion.mp4", fourcc, fps, (width, height))

#Se define la línea de referencia para contar los vehículos
#Las coordenadas se calculan proporcionalmente al tamaño del vídeo
linea_vehiculos_y = int(height * 430 / 928)
linea_vehiculos_x1 = int(width * 500 / 1650)
linea_vehiculos_x2 = width - 5

#Se define el segmento de referencia para contar los peatones
#Se sitúa sobre la acera y sigue aproximadamente su dirección
persona_x1 = int(width * 450 / 1650)
persona_y1 = int(height * 325 / 928)

persona_x2 = int(width * 500 / 1650)
persona_y2 = int(height * 390 / 928)

#Se establece un margen alrededor de la línea de vehículos para evitar
#contar cruces provocados por pequeñas oscilaciones de la detección
margen = 12

#Contadores de objetos según su dirección de movimiento
flujo = {"person": {"acercandose": 0, "alejandose": 0}, "bicycle": {"acercandose": 0, "alejandose": 0}, 
         "car": {"acercandose": 0, "alejandose": 0}}

#Se crean los siguientes diccionarios para almacenar la última posición conocida de cada objeto
posiciones = {}
posiciones_personas = {}

#Contados respresenta el conjunto de IDs que ya han sido contabilizados
contados = set()

while cap.isOpened():
    ret, img = cap.read()

    if not ret:
        break

    #Se detecta y realiza el seguimiento de personas, bicicletas y coches
    results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

    for box in results[0].boxes:

        #Solo se procesan detecciones que tengan un ID de seguimiento
        if box.id is None:
            continue

        #Se obbtiene las coordenadas de la caja, el ID y la clase del objeto
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        track_id = int(box.id[0])
        cls = int(box.cls[0])
        nombre = classNames[cls]

        #Se utiliza el centro inferior de la caja como referencia
        #para determinar la posición del objeto respecto a las líneas
        cx = (x1 + x2) // 2
        cy = y2

        #Se analiza la dirección de coches y bicicletas
        if nombre in ["car", "bicycle"]:

            #Se determina si el objeto se encuentra dentro del tramo horizontal
            #donde está definida la línea de referencia
            if linea_vehiculos_x1 <= cx <= linea_vehiculos_x2:

                #Se determina en qué lado de la línea se encuentra
                #El margen permite ignorar posiciones demasiado próximas
                if cy < linea_vehiculos_y - margen:
                    lado_actual = -1
                elif cy > linea_vehiculos_y + margen:
                    lado_actual = 1
                else:
                    lado_actual = 0

                #Se guarda la primera posición válida del objeto
                if track_id not in posiciones:
                    if lado_actual != 0:
                        posiciones[track_id] = lado_actual

                #Si el objeto todavía no se ha contado, comprobar
                #si ha cambiado de lado respecto a su posición anterior
                elif track_id not in contados:

                    lado_anterior = posiciones[track_id]

                    if lado_actual != 0:

                        #Un cambio de arriba hacia abajo indica que se acerca
                        if lado_anterior == -1 and lado_actual == 1:
                            flujo[nombre]["acercandose"] += 1
                            contados.add(track_id)

                        #Un cambio de abajo hacia arriba indica que se aleja
                        elif lado_anterior == 1 and lado_actual == -1:
                            flujo[nombre]["alejandose"] += 1
                            contados.add(track_id)

                        #Se actualiza el último lado conocido del objeto
                        posiciones[track_id] = lado_actual

        #Se analiza la dirección de los peatones
        elif nombre == "person":

            #Se guarda la posición inicial del peatón
            if track_id not in posiciones_personas:
                posiciones_personas[track_id] = (cx, cy)

            elif track_id not in contados:

                #Se recupera la posición anterior para compararla con la actual
                anterior_x, anterior_y = posiciones_personas[track_id]

                #Se calcula el lado de la línea en el que se encuentra
                #cada posición mediante el producto vectorial
                lado_anterior = ((persona_x2 - persona_x1) * (anterior_y - persona_y1) - (persona_y2 - persona_y1) * (anterior_x - persona_x1))

                lado_actual = ((persona_x2 - persona_x1) * (cy - persona_y1) - (persona_y2 - persona_y1) * (cx - persona_x1))

                #Si los resultados tienen signos opuestos,
                #el peatón ha pasado de un lado de la línea al otro
                if lado_anterior * lado_actual < 0:

                    #Se comprueba que el cruce se ha producido cerca
                    #del segmento definido para los peatones
                    medio_x = (anterior_x + cx) // 2
                    medio_y = (anterior_y + cy) // 2

                    cerca_linea = (
                        min(persona_x1, persona_x2) - 30 <= medio_x <= max(persona_x1, persona_x2) + 30
                        and min(persona_y1, persona_y2) - 30 <= medio_y <= max(persona_y1, persona_y2) + 30
                    )

                    if cerca_linea:
                        #Se clasifica el sentido del movimiento según
                        #el lado de la línea al que se ha desplazado
                        if lado_anterior > 0 and lado_actual < 0:
                            flujo["person"]["alejandose"] += 1

                        else:
                            flujo["person"]["acercandose"] += 1

                        contados.add(track_id)

                #Se actualiza la posición del peatón para el siguiente fotograma
                posiciones_personas[track_id] = (cx, cy)

        #Se dibuja la caja delimitadora y el identificador del objeto
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(img, f"{nombre} ID:{track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    #Se dibuja la línea roja de referencia para coches y bicicletas
    cv2.line(img, (linea_vehiculos_x1, linea_vehiculos_y), (linea_vehiculos_x2, linea_vehiculos_y), (0, 0, 255), 3)

    #Se dibujan los límites del margen de seguridad
    cv2.line(img, (linea_vehiculos_x1, linea_vehiculos_y - margen), (linea_vehiculos_x2, linea_vehiculos_y - margen), (0, 255, 255), 1)

    cv2.line(img, (linea_vehiculos_x1, linea_vehiculos_y + margen), (linea_vehiculos_x2, linea_vehiculos_y + margen), (0, 255, 255), 1)

    #Se dibuja la línea azul utilizada para analizar el movimiento de los peatones
    cv2.line(img, (persona_x1, persona_y1), (persona_x2, persona_y2), (255, 0, 0), 3)

    #Se muestran los contadores de cada clase y dirección en el fotograma
    y = 30

    for nombre in classNames:
        texto = (f"{nombre}: "f"acercandose {flujo[nombre]['acercandose']} "f"alejandose {flujo[nombre]['alejandose']}")

        cv2.putText(img, texto, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        y += 25

    #Se guarda y muestra el fotograma procesado
    out.write(img)
    cv2.imshow("Analisis de direccion", img)

    #Se detiene la ejecución si se pulsa ESC
    if cv2.waitKey(20) == 27:
        break

#Se liberan los recursos y cierran las ventanas
cap.release()
out.release()
cv2.destroyAllWindows()

**3. Entrenamiento del modelo de matrículas**

Se prepara y entrena un modelo YOLO11n para detectar matrículas a partir del conjunto de imágenes anotadas.

In [ ]:
#Se obtienen las rutas de las carpetas del proyecto
carpeta_p4 = Path.cwd()
carpeta_proyecto = carpeta_p4.parent

#Se definen las rutas del dataset, el modelo inicial y la carpeta de resultados
dataset_dir = carpeta_proyecto / "matriculas" / "dataset_yolo"
modelo_path = carpeta_p4 / "yolo11n.pt"
carpeta_runs = carpeta_proyecto / "matriculas" / "runs"

#Se selecciona la GPU si está disponible, o la CPU en caso contrario
device = 0 if torch.cuda.is_available() else "cpu"

print("Dispositivo:", device)

#Se carga el modelo YOLO11n que se utilizará para el entrenamiento
model = YOLO(str(modelo_path))

#Se entrena el modelo utilizando el dataset preparado
resultados = model.train(
    data=str(dataset_dir / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=640,
    batch=12,
    workers=0,
    device=device,
    project=str(carpeta_runs),
    name="train_val_matriculas",
    exist_ok=True,
    plots=True,
    save=True,
    save_period=10
)

**4. Evaluación del modelo**

Se evalúa el modelo entrenado utilizando el conjunto de test y se muestran las principales métricas obtenidas.

In [ ]:
#Se carga el mejor modelo obtenido durante el entrenamiento
best_model = YOLO("../matriculas/runs/train_val_matriculas/weights/best.pt")

#Se evalúa el modelo utilizando el conjunto de imágenes de test
resultados_test = best_model.val(
    data=str(dataset_dir / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device=device,
    plots=True,
    project=str(carpeta_runs),
    name="test_matriculas",
    exist_ok=True
)

#Se muestran las métricas obtenidas durante la evaluación
print("\nRESULTADOS DEL TEST")
print("-------------------")
print(f"Precision: {resultados_test.box.mp:.4f}")
print(f"Recall:    {resultados_test.box.mr:.4f}")
print(f"mAP50:     {resultados_test.box.map50:.4f}")
print(f"mAP50-95:  {resultados_test.box.map:.4f}")

#Se muestra la ruta donde se han guardado los resultados
print("\nResultados guardados en:", carpeta_runs)

**5. Procesamiento del vídeo y detección de matrículas**

Se procesa el vídeo aplicando detección y seguimiento de objetos. También se permite seleccionar entre la detección de matrículas mediante YOLO o mediante contornos, así como activar la anonimización y guardar los resultados en un archivo CSV.

In [ ]:
#Se pregunta si se desea detectar las matrículas mediante contornos
detectar_por_contornos = input("¿Detectar por contornos? (s/n): ").lower() == "s"

#Se cargan los modelos para detectar objetos y matrículas
model = YOLO('yolo11n.pt')  # Personas, bicicletas y coches

#Se cargan las opciones de anonimización y el modelo de matrículas cuando se utiliza YOLO
if not detectar_por_contornos:

    model_matriculas = YOLO("../matriculas/runs/train_val_matriculas/weights/best.pt")

    #Se pregunta si se desea anonimizar las matrículas y las personas
    anonimizar_matriculas = input("¿Anonimizar matrículas? (s/n): ").lower() == "s"
    anonimizar_personas = input("¿Anonimizar personas? (s/n): ").lower() == "s"

#Se define una función para anonimizar una región mediante desenfoque gaussiano
def anonimizar_region(img, x1, y1, x2, y2):

    #Se obtiene la región que se desea anonimizar
    region = img[y1:y2, x1:x2]

    #Se aplica un desenfoque gaussiano sobre la región
    region = cv2.GaussianBlur(region, (51, 51), 30)

    #Se actualiza la región original
    img[y1:y2, x1:x2] = region

    return img

#Se define una función para detectar matrículas mediante contornos
def detectar_matriculas_contornos(img, x_offset, y_offset):

    #Se convierte la imagen a escala de grises
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    #Se aplica un desenfoque gaussiano para reducir el ruido
    gris = cv2.GaussianBlur(gris, (5, 5), 0)

    #Se detectan los bordes mediante Canny
    bordes = cv2.Canny(gris, 70, 200)

    #Se define un elemento estructurante para unir bordes próximos horizontalmente
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 3))

    #Se aplica un cierre morfológico para unir los bordes de las posibles matrículas
    bordes = cv2.morphologyEx(bordes, cv2.MORPH_CLOSE, kernel)

    #Se extraen los contornos encontrados en la imagen
    contornos, _ = cv2.findContours(bordes, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    #Se prepara una lista para almacenar las posibles matrículas
    matriculas = []

    for contorno in contornos:

        #Se obtiene el rectángulo que delimita cada contorno
        x, y, w, h = cv2.boundingRect(contorno)

        #Se calcula el área y la relación entre anchura y altura
        area = w * h
        relacion = w / h if h > 0 else 0

        #Se filtran los contornos según sus dimensiones y proporciones
        if w >= 30 and h >= 10 and 2.5 <= relacion <= 5.5 and area >= 300:

            #Se comprueba que el contorno no ocupe una parte excesiva de la imagen
            if area <= img.shape[0] * img.shape[1] * 0.20:

                #Se ajustan las coordenadas a la posición original del fotograma
                mx1 = x + x_offset
                my1 = y + y_offset
                mx2 = mx1 + w
                my2 = my1 + h

                #Se almacena la información de la posible matrícula detectada
                matriculas.append({
                    "x1": mx1,
                    "y1": my1,
                    "x2": mx2,
                    "y2": my2,
                    "confianza": None,
                    "area": area,
                    "confianza_ocr": "None",
                    "texto": "None"
                })

    return matriculas

#Se definen las etiquetas correspondientes a las distintas clases detectadas
classNames = ["person", "bicycle", "car"]

#Se inicia la captura del vídeo
vid = cv2.VideoCapture("C0142.MP4")

track_history = defaultdict(lambda: [])

#Se obtiene la información del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

#Se prepara el archivo de vídeo donde se guardarán los fotogramas procesados
out = cv2.VideoWriter("C0142_procesado.mp4", fourcc, fps, (width, height))

#Se crea el archivo CSV para almacenar los resultados de las detecciones cuando se utiliza YOLO
if not detectar_por_contornos:

    archivo_csv = open("C0142_resultados.csv", "w", newline="", encoding="utf-8")
    campos = ["fotograma", "tipo_objeto", "confianza", "identificador_tracking", "x1", "y1", "x2", "y2",
        "matricula_en_su_caso", "confianza_matricula", "mx1", "my1", "mx2", "my2", "texto_matricula"
    ]

    escritor = csv.DictWriter(archivo_csv, fieldnames=campos)
    escritor.writeheader()

#Se inicializan los contadores y los conjuntos que almacenan los identificadores ya contabilizados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

#Se inicializa el contador de fotogramas
num_fotograma = 0

while(True):

    #Se obtiene el siguiente fotograma del vídeo
    ret, img = vid.read()

    #Se comprueba si se ha obtenido una imagen válida
    if ret:

        num_fotograma += 1

        #Se realiza el seguimiento de personas, bicicletas y coches
        results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

        #Se prepara una lista para almacenar las matrículas detectadas
        matriculas = []

        #Se detectan las matrículas mediante YOLO cuando no se ha seleccionado la detección por contornos
        if not detectar_por_contornos:

            results_matriculas = model_matriculas.predict(img, conf=0.25, verbose=False)

            for r_mat in results_matriculas:
                for box_mat in r_mat.boxes:

                    #Se obtienen las coordenadas de cada matrícula
                    mx1, my1, mx2, my2 = box_mat.xyxy[0].cpu().numpy()
                    mx1, my1, mx2, my2 = int(mx1), int(my1), int(mx2), int(my2)

                    #Se ajustan las coordenadas para evitar que queden fuera de la imagen
                    mx1 = max(0, mx1)
                    my1 = max(0, my1)
                    mx2 = min(width, mx2)
                    my2 = min(height, my2)

                    #Se obtiene la confianza de la detección de la matrícula
                    confianza_mat = float(box_mat.conf[0])

                    #Se anonimiza la matrícula cuando se ha activado la opción correspondiente
                    if anonimizar_matriculas:
                        img = anonimizar_region(img, mx1, my1, mx2, my2)

                    #Se almacena la información de la matrícula detectada
                    matriculas.append({"x1": mx1, "y1": my1, "x2": mx2, "y2": my2,
                        "confianza": confianza_mat, "area": (mx2 - mx1) * (my2 - my1),
                        "confianza_ocr": "None", "texto": "None"
                    })

        else:

            #Se buscan matrículas mediante contornos únicamente dentro de las regiones de los coches
            for r in results:

                for box in r.boxes:

                    #Se identifica la clase del objeto detectado
                    cls = int(box.cls[0])

                    if cls == 2:

                        #Se obtienen las coordenadas del coche detectado
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                        #Se ajustan las coordenadas para evitar que queden fuera de la imagen
                        x1 = max(0, x1)
                        y1 = max(0, y1)
                        x2 = min(width, x2)
                        y2 = min(height, y2)

                        #Se extrae la región del coche para buscar posibles matrículas
                        region_coche = img[y1:y2, x1:x2]

                        #Se comprueba que la región del coche tenga dimensiones válidas
                        if region_coche.size > 0:

                            #Se detectan los posibles contornos de las matrículas del coche
                            matriculas_detectadas = detectar_matriculas_contornos(region_coche, x1, y1)

                            #Se incorporan las matrículas detectadas a la lista general
                            matriculas.extend(matriculas_detectadas)

        #Se recorren las detecciones de personas, bicicletas y coches
        for r in results:

            boxes = r.boxes

            for box in boxes:

                #Se obtienen las coordenadas del objeto detectado
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                #Se ajustan las coordenadas para evitar que queden fuera de la imagen
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                #Se obtiene la confianza de la detección
                confidence = float(box.conf[0])
                print("Confianza =", round(confidence, 2))

                #Se identifica la clase del objeto detectado
                cls = int(box.cls[0])
                nombre_clase = classNames[cls]
                print("Clase =", nombre_clase)

                #Se obtiene el identificador asignado al objeto durante el seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0]))
                else:
                    track_id = ""

                #Se anonimiza la persona cuando se ha activado la opción correspondiente
                if not detectar_por_contornos and cls == 0 and anonimizar_personas:
                    img = anonimizar_region(img, x1, y1, x2, y2)

                #Se inicializa la información de la matrícula asociada al objeto
                matricula_asociada = None

                #Se buscan matrículas asociadas únicamente a las detecciones de coches
                if cls == 2:

                    for mat in matriculas:

                        #Se calcula el centro de la matrícula detectada
                        centro_x = (mat["x1"] + mat["x2"]) // 2
                        centro_y = (mat["y1"] + mat["y2"]) // 2

                        #Se comprueba si el centro de la matrícula se encuentra dentro del coche
                        if (x1 <= centro_x <= x2 and
                            y1 <= centro_y <= y2):

                            #Se selecciona la matrícula con mayor confianza cuando se utiliza YOLO
                            if not detectar_por_contornos:

                                if (matricula_asociada is None or mat["confianza"] > matricula_asociada["confianza"]):
                                    matricula_asociada = mat

                            #Se selecciona el contorno de mayor área cuando se utiliza detección por contornos
                            else:

                                if (matricula_asociada is None or mat["area"] > matricula_asociada["area"]):
                                    matricula_asociada = mat

                #Se actualizan los contadores cuando aparece un identificador que todavía no se ha contabilizado
                if box.id is not None:

                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                #Se prepara la información de la matrícula para almacenarla en el CSV
                if matricula_asociada is not None:

                    tiene_matricula = "Si"
                    confianza_matricula = matricula_asociada["confianza"]
                    mx1 = matricula_asociada["x1"]
                    my1 = matricula_asociada["y1"]
                    mx2 = matricula_asociada["x2"]
                    my2 = matricula_asociada["y2"]
                    texto_matricula = matricula_asociada["texto"]

                else:
                    tiene_matricula = "No"
                    confianza_matricula = ""
                    mx1 = ""
                    my1 = ""
                    mx2 = ""
                    my2 = ""
                    texto_matricula = ""

                #Se registra en el CSV la información del objeto y, si corresponde, de su matrícula
                if not detectar_por_contornos:

                    escritor.writerow({
                        "fotograma": num_fotograma, "tipo_objeto": nombre_clase,
                        "confianza": confidence, "identificador_tracking": track_id,
                        "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                        "matricula_en_su_caso": tiene_matricula,
                        "confianza_matricula": confianza_matricula,
                        "mx1": mx1, "my1": my1, "mx2": mx2, "my2": my2,
                        "texto_matricula": texto_matricula
                    })

                #Se asigna un color diferente a cada clase de objeto
                if cls == 0:
                    R = 128
                    G = 0
                    B = 0
                elif cls == 1:
                    R = 0
                    G = 128
                    B = 0
                elif cls == 2:
                    R = 0
                    G = 0
                    B = 128

                #Se dibuja el rectángulo y la etiqueta del objeto detectado
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)

                cv2.putText(img, track_id + ' ' + nombre_clase, [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

                #Se muestra la información de la matrícula cuando se ha asociado a un coche
                if matricula_asociada is not None:

                    texto = texto_matricula

                    if texto == "":
                        texto = "Matricula no reconocida"

                    cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 255), 2)

                    cv2.putText(img, texto, (mx1, max(20, my1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        #Se muestran los contadores de personas, bicicletas y coches en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        #Se guarda el fotograma procesado en el vídeo de salida
        out.write(img)

        #Se muestra el fotograma procesado
        cv2.imshow('Vid', img)

    else:
        break

    #Se comprueba si se ha pulsado ESC para detener el procesamiento
    if cv2.waitKey(20) == 27:
        break

#Se libera el objeto de captura del vídeo
vid.release()

#Se libera el objeto de escritura del vídeo
out.release()

#Se cierra el archivo CSV cuando se ha utilizado la detección mediante YOLO
if not detectar_por_contornos:
    archivo_csv.close()

#Se cierran las ventanas de visualización
cv2.destroyAllWindows()